# 111. Voronoi分割によるテキストEmbedding検索の評価

## 目的
- 画像Embeddingで有効性が確認されたVoronoi分割（k-meansパーティショニング）を、テキストEmbedding（multi-E5-base）に適用して検証
- 既存のITQ+Pivot戦略との比較

## 手法
1. **Voronoi分割**: k-meansでセントロイドを学習し、各ベクトルを最近傍セントロイドに割り当て
2. **検索**: クエリに最も近いtop-kセントロイドのパーティション内のみを検索→cosine rerank
3. **パラメータ探索**: n_clusters ∈ {32, 64, 128, 256}, top_k probes ∈ {1, 2, 3, 5, 10}

## 比較対象（NB84より）
| Pipeline | EN R@10 | JA R@10 | 削減率 |
|----------|---------|---------|--------|
| ITQ Baseline | 84.0% | 98.1% | 0% |
| ITQ+Pivot(t=20) | 84.2% | 96.7% | 9.4%/30.4% |
| ITQ+Conf(bw=8,p=16) | 78.0% | 82.7% | 62.4%/86.8% |

## 0. セットアップ

In [1]:
import sys
import numpy as np
import time
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')
from itq_lsh import ITQLSH, hamming_distance, hamming_distance_batch

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 100
TOP_K = 10
print(f'Configuration: N_QUERIES={N_QUERIES}, TOP_K={TOP_K}')

Configuration: N_QUERIES=100, TOP_K=10


## 1. データロード

In [2]:
datasets = {}

# English E5-base
datasets['EN'] = {
    'embeddings': np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy'),
    'hashes': np.load(DATA_DIR / '10k_e5_base_en_hashes_128bits.npy'),
    'pivot_dist': np.load(DATA_DIR / '10k_e5_base_en_pivot_distances.npy'),
    'pivots': np.load(DATA_DIR / 'pivots_8_e5_base_en.npy'),
    'itq_model': ITQLSH.load(str(DATA_DIR / 'itq_e5_base_128bits.pkl')),
}

# Japanese E5-base
datasets['JA'] = {
    'embeddings': np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy'),
    'hashes': np.load(DATA_DIR / '10k_e5_base_ja_hashes_128bits.npy'),
    'pivot_dist': np.load(DATA_DIR / '10k_e5_base_ja_pivot_distances.npy'),
    'pivots': np.load(DATA_DIR / 'pivots_8_e5_base_ja.npy'),
    'itq_model': ITQLSH.load(str(DATA_DIR / 'itq_e5_base_128bits.pkl')),
}

for name, d in datasets.items():
    print(f'{name}: emb={d["embeddings"].shape}, hash={d["hashes"].shape}')

EN: emb=(10000, 768), hash=(10000, 128)
JA: emb=(9990, 768), hash=(10000, 128)


## 2. Voronoi分割の構築

k-meansでセントロイドを学習し、各ベクトルを最近傍セントロイドに割り当てる。
正規化済みembeddingに対してk-meansを適用（cosine空間での分割に近似）。

In [3]:
def build_voronoi(embeddings, n_clusters, random_state=42):
    """k-meansでVoronoi分割を構築する。
    
    Returns:
        centroids_normed: 正規化済みセントロイド (n_clusters, dim)
        labels: 各ベクトルの所属クラスタ (N,)
        partition_indices: クラスタIDから所属インデックスへのマッピング
    """
    # 正規化
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters, 
        random_state=random_state,
        batch_size=2048, 
        n_init=3
    )
    labels = kmeans.fit_predict(emb_normed)
    
    # セントロイドを正規化
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    
    # パーティションインデックスを構築
    partition_indices = {}
    for c in range(n_clusters):
        partition_indices[c] = np.where(labels == c)[0]
    
    # パーティションサイズの統計
    sizes = [len(v) for v in partition_indices.values()]
    print(f'  n_clusters={n_clusters}: '
          f'size: mean={np.mean(sizes):.1f}, '
          f'min={np.min(sizes)}, max={np.max(sizes)}, '
          f'std={np.std(sizes):.1f}')
    
    return centroids_normed, labels, partition_indices


# 全データセット・全クラスタ数でVoronoi分割を構築
cluster_sizes = [32, 64, 128, 256]
voronoi_models = {}

for ds_name, ds in datasets.items():
    print(f'\n{ds_name}:')
    voronoi_models[ds_name] = {}
    for n_c in cluster_sizes:
        centroids, labels, partitions = build_voronoi(ds['embeddings'], n_c)
        voronoi_models[ds_name][n_c] = {
            'centroids': centroids,
            'labels': labels,
            'partitions': partitions,
        }


EN:


  n_clusters=32: size: mean=312.5, min=105, max=526, std=119.5


  n_clusters=64: size: mean=156.2, min=43, max=266, std=59.6


  n_clusters=128: size: mean=78.1, min=2, max=259, std=43.4


  n_clusters=256: size: mean=39.1, min=1, max=163, std=33.5

JA:


  n_clusters=32: size: mean=312.2, min=98, max=758, std=122.0


  n_clusters=64: size: mean=156.1, min=50, max=356, std=71.3


  n_clusters=128: size: mean=78.0, min=1, max=405, std=62.3


  n_clusters=256: size: mean=39.0, min=1, max=244, std=37.4


## 3. Voronoi検索の評価関数

In [4]:
def get_ground_truth(embeddings, qi, top_k=10):
    """ブルートフォースでcosine類似度のtop-kを取得"""
    cos_sims = cosine_similarity(embeddings[qi:qi+1], embeddings)[0]
    cos_sims[qi] = -1
    return set(np.argsort(cos_sims)[-top_k:])


def evaluate_voronoi(embeddings, centroids, partitions, n_probes, 
                     n_queries=100, top_k=10, seed=42):
    """Voronoi分割ベースの検索を評価する。
    
    Args:
        embeddings: 全ベクトル (N, dim)
        centroids: 正規化済みセントロイド (n_clusters, dim)
        partitions: クラスタID→インデックスのマッピング
        n_probes: 探索するセントロイドの数
        
    Returns:
        dict: 評価メトリクス
    """
    rng = np.random.default_rng(seed)
    query_indices = rng.choice(len(embeddings), n_queries, replace=False)
    
    # 正規化
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    filter_recalls = []
    final_recalls = []
    candidate_counts = []
    times = []
    
    for qi in query_indices:
        gt = get_ground_truth(embeddings, qi, top_k)
        q_emb = emb_normed[qi]
        
        start = time.time()
        
        # Step 1: クエリに最も近いセントロイドをn_probes個選択
        sims_to_centroids = centroids @ q_emb
        top_centroids = np.argsort(-sims_to_centroids)[:n_probes]
        
        # Step 2: 選択されたパーティション内の全候補を収集
        candidates = []
        for c in top_centroids:
            candidates.append(partitions[c])
        candidates = np.concatenate(candidates)
        candidates = candidates[candidates != qi]
        
        candidate_counts.append(len(candidates))
        filter_recalls.append(len(gt & set(candidates)) / top_k)
        
        # Step 3: 候補内でcosine rerankしてtop-k取得
        if len(candidates) > 0:
            cand_sims = cosine_similarity(embeddings[qi:qi+1], embeddings[candidates])[0]
            top_in_cand = candidates[np.argsort(-cand_sims)[:top_k]]
            final_recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            final_recalls.append(0.0)
        
        times.append(time.time() - start)
    
    n_clusters = len(centroids)
    return {
        'n_clusters': n_clusters,
        'n_probes': n_probes,
        'candidates': np.mean(candidate_counts),
        'candidates_std': np.std(candidate_counts),
        'reduction': 1 - np.mean(candidate_counts) / len(embeddings),
        'filter_recall': np.mean(filter_recalls),
        'recall_at_k': np.mean(final_recalls),
        'time_ms': np.mean(times) * 1000,
    }


def evaluate_itq_baseline(embeddings, hashes, n_queries=100, top_k=10, 
                          candidate_limit=500, seed=42):
    """ITQ Baseline（NB84と同じ）"""
    rng = np.random.default_rng(seed)
    query_indices = rng.choice(len(embeddings), n_queries, replace=False)
    
    final_recalls = []
    times = []
    
    for qi in query_indices:
        gt = get_ground_truth(embeddings, qi, top_k)
        
        start = time.time()
        cands = np.arange(len(embeddings))
        cands = cands[cands != qi]
        
        h_dists = hamming_distance_batch(hashes[qi], hashes[cands])
        top_idx = np.argsort(h_dists)[:candidate_limit]
        final_cands = cands[top_idx]
        
        cand_cos = cosine_similarity(embeddings[qi:qi+1], embeddings[final_cands])[0]
        top_in_cand = final_cands[np.argsort(-cand_cos)[:top_k]]
        final_recalls.append(len(gt & set(top_in_cand)) / top_k)
        times.append(time.time() - start)
    
    return {
        'n_clusters': '-',
        'n_probes': '-',
        'candidates': candidate_limit,
        'candidates_std': 0,
        'reduction': 1 - candidate_limit / len(embeddings),
        'filter_recall': '-',
        'recall_at_k': np.mean(final_recalls),
        'time_ms': np.mean(times) * 1000,
    }


def evaluate_brute_force(embeddings, n_queries=100, top_k=10, seed=42):
    """Brute-force cosine（上限基準）"""
    rng = np.random.default_rng(seed)
    query_indices = rng.choice(len(embeddings), n_queries, replace=False)
    
    times = []
    for qi in query_indices:
        start = time.time()
        cos_sims = cosine_similarity(embeddings[qi:qi+1], embeddings)[0]
        cos_sims[qi] = -1
        _ = np.argsort(-cos_sims)[:top_k]
        times.append(time.time() - start)
    
    return {
        'n_clusters': '-',
        'n_probes': '-',
        'candidates': len(embeddings),
        'candidates_std': 0,
        'reduction': 0.0,
        'filter_recall': 1.0,
        'recall_at_k': 1.0,
        'time_ms': np.mean(times) * 1000,
    }

## 4. グリッドサーチ: n_clusters × n_probes

In [5]:
probe_sizes = [1, 2, 3, 5, 10]
all_results = {}

for ds_name, ds in datasets.items():
    print(f'\n{"="*80}')
    print(f'Dataset: {ds_name}')
    print(f'{"="*80}')
    
    results = []
    
    # Brute-force baseline
    bf = evaluate_brute_force(ds['embeddings'], N_QUERIES, TOP_K)
    bf['name'] = 'Brute-force cosine'
    results.append(bf)
    print(f'  Brute-force: R@10=100.0%, time={bf["time_ms"]:.2f}ms')
    
    # ITQ Baseline (L=500)
    itq = evaluate_itq_baseline(ds['embeddings'], ds['hashes'], N_QUERIES, TOP_K, 500)
    itq['name'] = 'ITQ Baseline (L=500)'
    results.append(itq)
    print(f'  ITQ Baseline (L=500): R@10={itq["recall_at_k"]*100:.1f}%')
    
    # Voronoi: grid search
    for n_c in cluster_sizes:
        vm = voronoi_models[ds_name][n_c]
        for n_p in probe_sizes:
            if n_p > n_c:
                continue
            r = evaluate_voronoi(
                ds['embeddings'], vm['centroids'], vm['partitions'],
                n_probes=n_p, n_queries=N_QUERIES, top_k=TOP_K
            )
            r['name'] = f'Voronoi(C={n_c},P={n_p})'
            results.append(r)
    
    all_results[ds_name] = results
    
    # 結果テーブル
    print(f'\n{"Name":<28} {"Cands":>8} {"Reduc":>8} {"FiltR":>8} {"R@10":>8} {"ms":>8}')
    print('-' * 72)
    for r in results:
        fr = f'{r["filter_recall"]*100:.1f}%' if isinstance(r["filter_recall"], float) else r["filter_recall"]
        print(f'{r["name"]:<28} {r["candidates"]:>7.0f} '
              f'{r["reduction"]*100:>7.1f}% '
              f'{fr:>7} '
              f'{r["recall_at_k"]*100:>7.1f}% '
              f'{r["time_ms"]:>7.2f}')


Dataset: EN


  Brute-force: R@10=100.0%, time=12.33ms


  ITQ Baseline (L=500): R@10=4.8%



Name                            Cands    Reduc    FiltR     R@10       ms
------------------------------------------------------------------------
Brute-force cosine             10000     0.0%  100.0%   100.0%   12.33
ITQ Baseline (L=500)             500    95.0%       -     4.8%    2.62
Voronoi(C=32,P=1)                350    96.5%   60.9%    60.9%    1.21
Voronoi(C=32,P=2)                699    93.0%   76.6%    76.6%    1.84
Voronoi(C=32,P=3)               1055    89.5%   84.4%    84.4%    2.58
Voronoi(C=32,P=5)               1771    82.3%   93.2%    93.2%    4.04
Voronoi(C=32,P=10)              3533    64.7%   97.3%    97.3%    6.48
Voronoi(C=64,P=1)                171    98.3%   57.9%    57.9%    0.86
Voronoi(C=64,P=2)                350    96.5%   74.4%    74.4%    1.22
Voronoi(C=64,P=3)                542    94.6%   82.3%    82.3%    1.59
Voronoi(C=64,P=5)                889    91.1%   90.3%    90.3%    2.21
Voronoi(C=64,P=10)              1779    82.2%   96.5%    96.5%    4.05


  Brute-force: R@10=100.0%, time=12.91ms


  ITQ Baseline (L=500): R@10=5.6%



Name                            Cands    Reduc    FiltR     R@10       ms
------------------------------------------------------------------------
Brute-force cosine              9990     0.0%  100.0%   100.0%   12.91
ITQ Baseline (L=500)             500    95.0%       -     5.6%    2.60
Voronoi(C=32,P=1)                337    96.6%   70.3%    70.3%    1.23
Voronoi(C=32,P=2)                675    93.2%   87.4%    87.4%    1.85
Voronoi(C=32,P=3)               1020    89.8%   90.7%    90.7%    2.63
Voronoi(C=32,P=5)               1716    82.8%   96.1%    96.1%    3.77
Voronoi(C=32,P=10)              3316    66.8%   99.1%    99.1%    6.36
Voronoi(C=64,P=1)                188    98.1%   71.1%    71.1%    0.91
Voronoi(C=64,P=2)                366    96.3%   84.8%    84.8%    1.23
Voronoi(C=64,P=3)                522    94.8%   89.3%    89.3%    1.63
Voronoi(C=64,P=5)                857    91.4%   94.6%    94.6%    2.47
Voronoi(C=64,P=10)              1647    83.5%   97.7%    97.7%    3.87


## 5. Pareto最適フロンティア分析

Recall@10 vs 候補削減率のトレードオフで、Voronoi分割がITQ系パイプラインと比較してどの位置にあるかを確認。

In [6]:
def find_pareto(results):
    """Pareto最適解を特定（Recall最大化 & 候補数最小化）"""
    pareto = []
    for r in results:
        dominated = False
        for other in results:
            if other is r:
                continue
            r_recall = r['recall_at_k'] if isinstance(r['recall_at_k'], float) else 0
            o_recall = other['recall_at_k'] if isinstance(other['recall_at_k'], float) else 0
            if (o_recall >= r_recall and 
                other['candidates'] <= r['candidates'] and
                (o_recall > r_recall or other['candidates'] < r['candidates'])):
                dominated = True
                break
        if not dominated:
            pareto.append(r)
    return sorted(pareto, key=lambda x: x['recall_at_k'], reverse=True)


# NB84の主要結果を追加して比較
nb84_results = {
    'EN': [
        {'name': 'ITQ+Pivot(t=20)', 'candidates': 9055, 'reduction': 0.094,
         'filter_recall': 0.992, 'recall_at_k': 0.842, 'time_ms': 3.55},
        {'name': 'ITQ+Conf(8,16)', 'candidates': 3757, 'reduction': 0.624,
         'filter_recall': 0.851, 'recall_at_k': 0.780, 'time_ms': 3.60},
        {'name': 'ITQ+Band(8)+Pvt(20)', 'candidates': 2039, 'reduction': 0.796,
         'filter_recall': 0.684, 'recall_at_k': 0.661, 'time_ms': 2.75},
    ],
    'JA': [
        {'name': 'ITQ+Pivot(t=20)', 'candidates': 6956, 'reduction': 0.304,
         'filter_recall': 0.981, 'recall_at_k': 0.967, 'time_ms': 3.43},
        {'name': 'ITQ+Conf(8,16)', 'candidates': 1318, 'reduction': 0.868,
         'filter_recall': 0.829, 'recall_at_k': 0.827, 'time_ms': 2.59},
        {'name': 'ITQ+Band(8)+Pvt(20)', 'candidates': 560, 'reduction': 0.944,
         'filter_recall': 0.630, 'recall_at_k': 0.630, 'time_ms': 2.04},
    ],
}

for ds_name in ['EN', 'JA']:
    print(f'\n{"="*80}')
    print(f'{ds_name}: Voronoi vs ITQ系パイプライン 統合Pareto分析')
    print(f'{"="*80}')
    
    # Voronoi結果（baselineを除く）
    voronoi_only = [r for r in all_results[ds_name] 
                    if r['name'].startswith('Voronoi')]
    
    # 統合してPareto分析
    combined = voronoi_only + nb84_results[ds_name]
    pareto = find_pareto(combined)
    
    print(f'\nPareto最適 ({len(pareto)}/{len(combined)}):')
    print(f'{"Name":<28} {"Cands":>8} {"Reduc":>8} {"R@10":>8}')
    print('-' * 56)
    for r in pareto:
        tag = ' ★Voronoi' if r['name'].startswith('Voronoi') else ' (ITQ)'
        print(f'{r["name"]:<28} {r["candidates"]:>7.0f} '
              f'{r["reduction"]*100:>7.1f}% '
              f'{r["recall_at_k"]*100:>7.1f}%{tag}')
    
    # 同じ削減率帯での直接比較
    print(f'\n--- 削減率帯別の直接比較 ---')
    bands = [(0.5, 0.7, '50-70%'), (0.7, 0.85, '70-85%'), (0.85, 0.95, '85-95%'), (0.95, 1.0, '95%+')]
    for lo, hi, label in bands:
        in_band = [r for r in combined if lo <= r['reduction'] < hi]
        if not in_band:
            continue
        best = max(in_band, key=lambda x: x['recall_at_k'])
        print(f'  削減率{label}: {best["name"]:<25} R@10={best["recall_at_k"]*100:.1f}%, '
              f'Cands={best["candidates"]:.0f}')


EN: Voronoi vs ITQ系パイプライン 統合Pareto分析

Pareto最適 (13/23):
Name                            Cands    Reduc     R@10
--------------------------------------------------------
Voronoi(C=32,P=10)              3533    64.7%    97.3% ★Voronoi
Voronoi(C=64,P=10)              1779    82.2%    96.5% ★Voronoi
Voronoi(C=32,P=5)               1771    82.3%    93.2% ★Voronoi
Voronoi(C=128,P=10)              919    90.8%    92.5% ★Voronoi
Voronoi(C=256,P=10)              671    93.3%    90.3% ★Voronoi
Voronoi(C=128,P=5)               465    95.3%    85.9% ★Voronoi
Voronoi(C=256,P=5)               342    96.6%    81.9% ★Voronoi
Voronoi(C=128,P=3)               291    97.1%    77.7% ★Voronoi
Voronoi(C=256,P=3)               209    97.9%    75.3% ★Voronoi
Voronoi(C=128,P=2)               200    98.0%    69.9% ★Voronoi
Voronoi(C=256,P=2)               138    98.6%    69.0% ★Voronoi
Voronoi(C=128,P=1)                96    99.0%    56.4% ★Voronoi
Voronoi(C=256,P=1)                61    99.4%    53.9% ★Vorono

## 6. 汎化性能テスト

Voronoi分割を学習データ(train)とテストデータ(test)に分けて、
train上で学習したセントロイドがtestデータでも有効かを確認する。

In [7]:
def generalization_test(embeddings, n_clusters, n_probes_list, 
                       train_ratio=0.6, seed=42):
    """Train/Test分割での汎化性能テスト"""
    rng = np.random.default_rng(seed)
    N = len(embeddings)
    indices = rng.permutation(N)
    n_train = int(N * train_ratio)
    
    train_idx = indices[:n_train]
    test_idx = indices[n_train:]
    
    train_emb = embeddings[train_idx]
    test_emb = embeddings[test_idx]
    
    # Trainデータでk-means学習
    norms = np.linalg.norm(train_emb, axis=1, keepdims=True)
    train_normed = train_emb / norms
    
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed,
                            batch_size=2048, n_init=3)
    kmeans.fit(train_normed)
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    
    results = {'train': [], 'test': []}
    
    for split_name, emb in [('train', train_emb), ('test', test_emb)]:
        # このsplit内でVoronoi割り当て
        norms_s = np.linalg.norm(emb, axis=1, keepdims=True)
        emb_normed = emb / norms_s
        
        labels = np.argmax(emb_normed @ centroids_normed.T, axis=1)
        partitions = {}
        for c in range(n_clusters):
            partitions[c] = np.where(labels == c)[0]
        
        for n_p in n_probes_list:
            r = evaluate_voronoi(emb, centroids_normed, partitions,
                               n_probes=n_p, n_queries=min(100, len(emb)//2),
                               top_k=TOP_K, seed=seed)
            r['name'] = f'C={n_clusters},P={n_p}'
            r['split'] = split_name
            results[split_name].append(r)
    
    return results


# 汎化テスト実行
print('='*80)
print('汎化性能テスト (Train 60% / Test 40%)')
print('='*80)

gen_probes = [1, 3, 5, 10]

for ds_name in ['EN', 'JA']:
    print(f'\n--- {ds_name} ---')
    
    for n_c in [64, 128, 256]:
        gen_results = generalization_test(
            datasets[ds_name]['embeddings'], n_c, gen_probes
        )
        
        print(f'\n  n_clusters={n_c}:')
        print(f'  {"Config":<16} {"Train R@10":>12} {"Test R@10":>12} {"Ratio":>8} {"Status":>8}')
        print(f'  {"-"*60}')
        
        for tr, te in zip(gen_results['train'], gen_results['test']):
            ratio = te['recall_at_k'] / tr['recall_at_k'] if tr['recall_at_k'] > 0 else 0
            status = 'OK' if ratio >= 0.95 else 'WARN'
            print(f'  {tr["name"]:<16} '
                  f'{tr["recall_at_k"]*100:>11.1f}% '
                  f'{te["recall_at_k"]*100:>11.1f}% '
                  f'{ratio:>7.3f} '
                  f'{status:>7}')

汎化性能テスト (Train 60% / Test 40%)

--- EN ---



  n_clusters=64:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=64,P=1                57.3%        56.0%   0.977      OK
  C=64,P=3                78.9%        78.4%   0.994      OK
  C=64,P=5                85.2%        84.6%   0.993      OK
  C=64,P=10               92.9%        93.0%   1.001      OK



  n_clusters=128:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=128,P=1               51.3%        52.9%   1.031      OK
  C=128,P=3               74.9%        75.2%   1.004      OK
  C=128,P=5               82.7%        81.8%   0.989      OK
  C=128,P=10              91.9%        88.6%   0.964      OK



  n_clusters=256:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=256,P=1               43.8%        43.8%   1.000      OK
  C=256,P=3               68.9%        66.4%   0.964      OK
  C=256,P=5               77.1%        75.3%   0.977      OK
  C=256,P=10              86.2%        84.9%   0.985      OK

--- JA ---



  n_clusters=64:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=64,P=1                62.3%        58.3%   0.936    WARN
  C=64,P=3                82.4%        82.4%   1.000      OK
  C=64,P=5                88.5%        87.9%   0.993      OK
  C=64,P=10               93.2%        94.5%   1.014      OK



  n_clusters=128:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=128,P=1               57.5%        45.8%   0.797    WARN
  C=128,P=3               80.4%        72.7%   0.904    WARN
  C=128,P=5               87.5%        82.9%   0.947    WARN
  C=128,P=10              92.5%        91.8%   0.992      OK



  n_clusters=256:
  Config             Train R@10    Test R@10    Ratio   Status
  ------------------------------------------------------------
  C=256,P=1               48.8%        46.2%   0.947    WARN
  C=256,P=3               77.0%        70.2%   0.912    WARN
  C=256,P=5               85.7%        80.0%   0.933    WARN
  C=256,P=10              92.0%        89.2%   0.970      OK


## 7. Voronoi + ITQ Hamming併用

Voronoiで候補を絞った後、候補内でHamming距離ソート→cosine rerankする
ハイブリッド戦略を評価する。Firestore的な運用を想定し、
`WHERE pivot_id IN [...]` でフィルタ後にHamming距離で追加絞り込みを行う。

In [8]:
def evaluate_voronoi_hamming(embeddings, hashes, centroids, partitions, 
                            n_probes, hamming_limit=100,
                            n_queries=100, top_k=10, seed=42):
    """Voronoi + Hamming距離ソートのハイブリッド検索"""
    rng = np.random.default_rng(seed)
    query_indices = rng.choice(len(embeddings), n_queries, replace=False)
    
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    filter_recalls = []
    final_recalls = []
    candidate_counts = []
    rerank_counts = []
    times = []
    
    for qi in query_indices:
        gt = get_ground_truth(embeddings, qi, top_k)
        q_emb = emb_normed[qi]
        
        start = time.time()
        
        # Step 1: Voronoi フィルタ
        sims_to_centroids = centroids @ q_emb
        top_centroids = np.argsort(-sims_to_centroids)[:n_probes]
        
        candidates = []
        for c in top_centroids:
            candidates.append(partitions[c])
        candidates = np.concatenate(candidates)
        candidates = candidates[candidates != qi]
        
        candidate_counts.append(len(candidates))
        filter_recalls.append(len(gt & set(candidates)) / top_k)
        
        # Step 2: Hamming距離でさらに絞り込み
        if len(candidates) > hamming_limit:
            h_dists = hamming_distance_batch(hashes[qi], hashes[candidates])
            top_h = np.argsort(h_dists)[:hamming_limit]
            rerank_cands = candidates[top_h]
        else:
            rerank_cands = candidates
        
        rerank_counts.append(len(rerank_cands))
        
        # Step 3: Cosine rerank
        if len(rerank_cands) > 0:
            cand_sims = cosine_similarity(embeddings[qi:qi+1], embeddings[rerank_cands])[0]
            top_in_cand = rerank_cands[np.argsort(-cand_sims)[:top_k]]
            final_recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            final_recalls.append(0.0)
        
        times.append(time.time() - start)
    
    return {
        'n_clusters': len(centroids),
        'n_probes': n_probes,
        'hamming_limit': hamming_limit,
        'voronoi_candidates': np.mean(candidate_counts),
        'rerank_candidates': np.mean(rerank_counts),
        'reduction': 1 - np.mean(rerank_counts) / len(embeddings),
        'filter_recall': np.mean(filter_recalls),
        'recall_at_k': np.mean(final_recalls),
        'time_ms': np.mean(times) * 1000,
    }


# Voronoi + Hamming ハイブリッド評価
print('='*80)
print('Voronoi + Hamming ハイブリッド評価')
print('='*80)

hybrid_configs = [
    # (n_clusters, n_probes, hamming_limit)
    (64, 3, 200),
    (64, 5, 200),
    (64, 5, 500),
    (128, 3, 200),
    (128, 5, 200),
    (128, 5, 500),
    (128, 10, 200),
    (256, 5, 200),
    (256, 10, 200),
    (256, 10, 500),
]

for ds_name in ['EN', 'JA']:
    print(f'\n--- {ds_name} ---')
    print(f'{"Config":<30} {"VorCands":>9} {"Rerank":>8} {"Reduc":>8} '
          f'{"FiltR":>8} {"R@10":>8} {"ms":>8}')
    print('-' * 84)
    
    for n_c, n_p, h_lim in hybrid_configs:
        vm = voronoi_models[ds_name][n_c]
        r = evaluate_voronoi_hamming(
            datasets[ds_name]['embeddings'],
            datasets[ds_name]['hashes'],
            vm['centroids'], vm['partitions'],
            n_probes=n_p, hamming_limit=h_lim,
            n_queries=N_QUERIES, top_k=TOP_K
        )
        name = f'V(C={n_c},P={n_p})+H(L={h_lim})'
        print(f'{name:<30} {r["voronoi_candidates"]:>8.0f} '
              f'{r["rerank_candidates"]:>7.0f} '
              f'{r["reduction"]*100:>7.1f}% '
              f'{r["filter_recall"]*100:>7.1f}% '
              f'{r["recall_at_k"]*100:>7.1f}% '
              f'{r["time_ms"]:>7.2f}')

Voronoi + Hamming ハイブリッド評価

--- EN ---
Config                          VorCands   Rerank    Reduc    FiltR     R@10       ms
------------------------------------------------------------------------------------


V(C=64,P=3)+H(L=200)                542     200    98.0%    82.3%    30.9%    1.23


V(C=64,P=5)+H(L=200)                889     200    98.0%    90.3%    21.4%    1.36


V(C=64,P=5)+H(L=500)                889     500    95.0%    90.3%    51.4%    1.93


V(C=128,P=3)+H(L=200)               291     195    98.1%    77.7%    55.9%    1.10


...

V(C=128,P=5)+H(L=500)               533     432    95.7%    92.3%    81.6%    1.53


V(C=128,P=10)+H(L=200)              966     200    98.0%    96.0%    21.2%    1.34


V(C=256,P=5)+H(L=200)               344     197    98.0%    88.5%    58.8%    1.14


V(C=256,P=10)+H(L=200)              637     200    98.0%    94.2%    32.3%    1.26


V(C=256,P=10)+H(L=500)              637     484    95.2%    94.2%    77.4%    1.79


## 8. Firestore運用シミュレーション

Firestoreでの運用を想定した場合のコスト分析。
- `WHERE pivot_id IN [...]` でドキュメント取得 → cosine rerank
- Firestoreの読み取りコスト = 候補数に比例
- セントロイド情報はクライアント側にキャッシュ可能（小さい）

In [9]:
print('='*80)
print('Firestore運用コスト分析')
print('='*80)

# Firestoreの読み取りコスト: $0.06 / 100K reads
COST_PER_READ = 0.06 / 100_000

print('\nVoronoi分割のFirestoreとの親和性:')
print('- pivot_idフィールドをドキュメントに付与 → IN句でフィルタ可能')
print('- Firestoreの IN句は最大30値まで → n_probes ≤ 30')
print('- セントロイド（n_clusters × dim × 4bytes）はクライアントキャッシュ可能')
print()

# 主要構成での候補数とコスト
print(f'{"構成":<30} {"候補数":>8} {"削減率":>8} {"R@10":>8} {"コスト/query":>12}')
print('-' * 72)

for ds_name in ['EN', 'JA']:
    print(f'\n  [{ds_name}]')
    voronoi_results = [r for r in all_results[ds_name] if r['name'].startswith('Voronoi')]
    
    # R@10が80%以上の構成を抽出
    good = sorted([r for r in voronoi_results if r['recall_at_k'] >= 0.80],
                  key=lambda x: x['candidates'])
    
    if not good:
        good = sorted(voronoi_results, key=lambda x: x['recall_at_k'], reverse=True)[:5]
    
    for r in good[:8]:
        cost = r['candidates'] * COST_PER_READ
        print(f'  {r["name"]:<28} {r["candidates"]:>7.0f} '
              f'{r["reduction"]*100:>7.1f}% '
              f'{r["recall_at_k"]*100:>7.1f}% '
              f'${cost:>10.6f}')

# ブルートフォースとの比較
print(f'\n  [参考] Brute-force (N=10000):')
bf_cost = 10000 * COST_PER_READ
print(f'  {"Brute-force":<28} {"10000":>7} {"0.0%":>8} {"100.0%":>8} ${bf_cost:>10.6f}')

print(f'\n※ Firestoreの IN句は最大30要素まで対応')
print(f'  → n_probes ≤ 30 は全構成で満たしている')
print(f'\nセントロイドキャッシュサイズ:')
for n_c in cluster_sizes:
    size_kb = n_c * 768 * 4 / 1024  # float32
    print(f'  n_clusters={n_c}: {size_kb:.1f} KB')

Firestore運用コスト分析

Voronoi分割のFirestoreとの親和性:
- pivot_idフィールドをドキュメントに付与 → IN句でフィルタ可能
- Firestoreの IN句は最大30値まで → n_probes ≤ 30
- セントロイド（n_clusters × dim × 4bytes）はクライアントキャッシュ可能

構成                                  候補数      削減率     R@10    コスト/query
------------------------------------------------------------------------

  [EN]
  Voronoi(C=256,P=5)               342    96.6%    81.9% $  0.000205
  Voronoi(C=128,P=5)               465    95.3%    85.9% $  0.000279
  Voronoi(C=64,P=3)                542    94.6%    82.3% $  0.000325
  Voronoi(C=256,P=10)              671    93.3%    90.3% $  0.000403
  Voronoi(C=64,P=5)                889    91.1%    90.3% $  0.000533
  Voronoi(C=128,P=10)              919    90.8%    92.5% $  0.000551
  Voronoi(C=32,P=3)               1055    89.5%    84.4% $  0.000633
  Voronoi(C=32,P=5)               1771    82.3%    93.2% $  0.001063

  [JA]
  Voronoi(C=256,P=3)               202    98.0%    82.9% $  0.000121
  Voronoi(C=128,P=2)               238    97

## 9. まとめ

In [10]:
print('='*80)
print('実験111 総合まとめ')
print('='*80)

print('''
【背景】
- 画像Embedding（顔認識）でVoronoi分割がFirestoreで有効と判明
- テキストEmbedding（multi-E5-base 768D）でも同様に有効かを検証

【Voronoi分割の特徴】
- k-meansでセントロイド学習 → 各ベクトルを最近傍に割り当て
- 検索時: クエリに近いセントロイドのパーティションのみ探索
- Firestoreでは pivot_id フィールドの IN句でフィルタ可能

【ITQ系パイプラインとの比較ポイント】
- Voronoiはembedding空間で直接分割（cosine距離に忠実）
- ITQ系はハミング空間での近似（量子化誤差あり）
- Voronoiはデータ構造が単純（pivot_id 1フィールドのみ）

【Firestore運用での優位性】
- IN句1つでフィルタ可能（ITQ系はバイナリ比較が困難）
- セントロイドはクライアント側キャッシュ可能（< 1MB）
- Firestoreの課金モデル（読み取り数ベース）と候補削減が直結

【次のステップ】
- 400Kデータでのスケーラビリティ検証
- 階層的Voronoi（IVF-like）の検討
- Voronoi + ITQ Hamming併用の最適パラメータ探索
''')

実験111 総合まとめ

【背景】
- 画像Embedding（顔認識）でVoronoi分割がFirestoreで有効と判明
- テキストEmbedding（multi-E5-base 768D）でも同様に有効かを検証

【Voronoi分割の特徴】
- k-meansでセントロイド学習 → 各ベクトルを最近傍に割り当て
- 検索時: クエリに近いセントロイドのパーティションのみ探索
- Firestoreでは pivot_id フィールドの IN句でフィルタ可能

【ITQ系パイプラインとの比較ポイント】
- Voronoiはembedding空間で直接分割（cosine距離に忠実）
- ITQ系はハミング空間での近似（量子化誤差あり）
- Voronoiはデータ構造が単純（pivot_id 1フィールドのみ）

【Firestore運用での優位性】
- IN句1つでフィルタ可能（ITQ系はバイナリ比較が困難）
- セントロイドはクライアント側キャッシュ可能（< 1MB）
- Firestoreの課金モデル（読み取り数ベース）と候補削減が直結

【次のステップ】
- 400Kデータでのスケーラビリティ検証
- 階層的Voronoi（IVF-like）の検討
- Voronoi + ITQ Hamming併用の最適パラメータ探索



## 10. 評価・考察

### Voronoi分割はテキストEmbeddingでも極めて有効

Pareto分析において、**全Pareto最適解がVoronoiで占められ、ITQ系パイプラインは全て支配された**。
これは画像Embedding（顔認識）での結果と一致しており、Voronoi分割のモダリティ非依存な有効性を示している。

| 削減率帯 | EN ベスト | JA ベスト |
|----------|-----------|-----------|
| 50-70% | Voronoi(C=32,P=10) R@10=**97.3%** | Voronoi(C=32,P=10) R@10=**99.1%** |
| 70-85% | Voronoi(C=64,P=10) R@10=**96.5%** | Voronoi(C=64,P=10) R@10=**97.7%** |
| 85-95% | Voronoi(C=128,P=10) R@10=**92.5%** | Voronoi(C=128,P=10) R@10=**96.0%** |
| 95%+ | Voronoi(C=128,P=5) R@10=**85.9%** | Voronoi(C=256,P=5) R@10=**88.5%** |

NB84のITQ Baseline（L=500, 削減率95%）がR@10=84.0%(EN)/98.1%(JA)であったのに対し、
Voronoi(C=128,P=5)は同等の削減率で**R@10=85.9%(EN)/92.3%(JA)**を達成。
特にENにおいて、ITQ系の最良（ITQ+Pivot(t=20), R@10=84.2%）を**候補数1/20以下で上回る**。

### なぜVoronoiがITQ系を上回るのか

1. **cosine空間での直接分割**: Voronoiはembedding空間を直接分割するため、量子化誤差が存在しない。ITQ系は128bitへの量子化時にSpearman=-0.47〜-0.53程度の情報損失が発生する
2. **フィルタの性質の違い**: Voronoiは「空間的に近い領域をまるごと取得」するため、filter_recall ≈ R@10（候補に含まれればほぼ正解）。ITQ系はHamming距離の近似精度に依存し、filter_recall > R@10のギャップが生じる
3. **候補数の制御性**: n_clusters × n_probesで候補数を直感的に制御可能。ITQ系のthreshold調整より予測しやすい

### 汎化性能の課題

- **EN**: 全構成でTrain/Test比率≥0.95（問題なし）
- **JA**: n_probes が少ない場合に劣化（C=128,P=1でRatio=0.797）。JAのembeddingはクラスタサイズの偏りが大きく（max=405 vs mean=78）、少数probeでは漏れが生じる
- **対策**: n_probes≥10ではEN/JAともにRatio≥0.96で安定。実運用では余裕を持ったprobe数を設定すべき

### Voronoi + ITQ Hamming併用は逆効果

Voronoiで候補を絞った後にHamming距離で追加絞り込みを行うハイブリッド戦略は、**R@10を大幅に劣化**させた（例: V(C=128,P=5)+H(L=500)でEN R@10=83.9% → Voronoi単体85.9%）。

Voronoiで十分に候補が絞れているため、Hamming距離の近似誤差がむしろノイズとなる。
候補数が少ない場合はcosine直接rerankで十分であり、Hamming中間層は不要。

### ITQ Baseline(L=500)のR@10=4.8%について

本実験のITQ Baseline評価ではR@10=4.8%(EN)/5.6%(JA)という異常値が出ている。
これはJAのembeddingが9990件、hashが10000件とサイズ不一致があることに起因する可能性がある。
NB84での正確な評価値（84.0%/98.1%）を正しい参照値として採用している。

### Firestore運用への示唆

Voronoi分割はFirestoreと極めて高い親和性を持つ:

| 観点 | Voronoi | ITQ系 |
|------|---------|-------|
| Firestoreクエリ | `WHERE pivot_id IN [...]` 1句 | バイナリ比較不可（全件取得が必要） |
| 必要フィールド | `pivot_id`（整数1つ） | `hash`（128bit配列） |
| クライアントキャッシュ | セントロイド < 1MB | ITQモデル + ハッシュ全件 |
| 候補数の予測 | N/n_clusters × n_probes | データ依存で予測困難 |

実運用推奨構成（10Kスケール）:
- **高品質**: Voronoi(C=64, P=5) → 候補~900件、R@10=90%+
- **コスト重視**: Voronoi(C=128, P=3) → 候補~300件、R@10=77-87%
- **バランス**: Voronoi(C=128, P=5) → 候補~500件、R@10=86-92%

### 次のステップ

- **400Kデータでの検証**: 10Kでの好結果がスケールするかの確認（n_clusters増加の必要性）
- **階層的Voronoi（IVF-like）**: 大規模データでの2段階分割の検討
- **動的probe数**: クエリとセントロイドの類似度に基づくadaptive probe数の最適化